# Brainstorming and Focus Group Quantitative Experimentation 2.3: **Difficult people** under **action correction** only

Can we use TinyTroupe to brainstorm product ideas?

In [1]:
import sys

from pprint import pprint

from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld
from tinytroupe.experimentation import InPlaceExperimentRunner
from tinytroupe.steering import Intervention
from tinytroupe.examples import *
from tinytroupe.validation import propositions
from tinytroupe.extraction import ResultsExtractor
from tinytroupe.utils.parallel import parallel_map_dict, parallel_map_cross
from tinytroupe.validation import hard_persona_adherence, persona_adherence, self_consistency, fluency, task_completion, divergence

# specific utilities
from common_utils import *


!!!!
DISCLAIMER: TinyTroupe relies on Artificial Intelligence (AI) models to generate content. 
The AI models are not perfect and may produce inappropriate or inaccurate results. 
For any serious or consequential use, please review the generated content before using it.
!!!!

Looking for default config on: C:\Users\pdasilva\repos\TinyTroupe and personal repos\TinyTroupe\tinytroupe\utils\..\config.ini
Found custom config on: c:\Users\pdasilva\repos\TinyTroupe and personal repos\TinyTroupe\publications\paper_artifacts_april-2026\config.ini
TinyTroupe version: 0.8.0
Current date and time (local): 2026-05-04 19:29:33
Current date and time (UTC):   2026-05-04 22:29:33

Current TinyTroupe configuration 
[OpenAI]
api_type = azure
azure_api_version = 2024-12-01-preview
model = gpt-5-mini
reasoning_model = o3-mini
vision_detail = auto
embedding_model = text-embedding-3-small
azure_embedding_model_api_version = 2023-05-15
max_completion_tokens = 128000
timeout = 300
max_attempts = 5
waiting_tim

## Parameters

In [2]:
full_mode = True  # set to True to run the full mode with all agents and tasks

# avoid displaying the communication, to make the output cleaner for eval
TinyPerson.communication_display = False

In [3]:
if full_mode:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 12
    qty_proposals = 4

else:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 4
    qty_proposals = 1


## Experiment setup

In [4]:
experiment_runner = InPlaceExperimentRunner("./brainstorming_and_focus_group_quantitative_experimentation_2.3.json")

experiment_runner.add_experiment("Control")
experiment_runner.add_experiment("Treatment")

2026-05-04 19:36:07,790 - MainThread(44240) - tinytroupe - WARNING - Configuration file './brainstorming_and_focus_group_quantitative_experimentation_2.3.json' exists and was loaded successfully. If you are trying to fully rerun the experiments, delete it first.
2026-05-04 19:36:07,803 - MainThread(44240) - tinytroupe - INFO - Experiment 'Control' already exists, nothihg to add.
2026-05-04 19:36:07,807 - MainThread(44240) - tinytroupe - INFO - Experiment 'Treatment' already exists, nothihg to add.


In [5]:
experiment_runner.activate_next_experiment()

#experiment_runner.fix_active_experiment("Control")
#experiment_runner.fix_active_experiment("Treatment")

In [6]:
print(f"Running experiment {experiment_runner.get_active_experiment()}")

Running experiment Treatment


## Agents and populations

In [7]:

people = []
if not experiment_runner.has_finished_all_experiments():
    # load agents
    people = TinyPerson.load_specifications_from_folder("./population/difficult_people_2")

    # filter to make it go faster?
    if qty_agents is not None:
        people = people[:qty_agents]

    # customize and print minibios 
    for person in people:
        person.import_fragment("./fragments/difficult_person.agent.fragment.json")
        print(person.minibio(extended=False))


Alan Merrick is a 48 year old Administrative Officer (Benefits and Records), British, currently living in Manchester, United Kingdom.
Anthony Russo is a 42 year old Journeyman Electrician / Senior Field Technician, American, currently living in Cleveland, Ohio, USA.
Anya Calder-Mori is a 45 year old Freelance Graphic Designer, Conceptual Artist and Cultural Critic, British, currently living in Camberwell, London, UK.
Barbara Jean Pratt is a 68 year old Retiree (former assembly line worker / part-time volunteer at church thrift shop), American, currently living in Small town near Toledo, Ohio, USA.
Colin Arthur Matthews is a 42 year old Operations Manager (Mid-level), British, currently living in Manchester, UK.
Colin Murray is a 52 year old Benefits and Housing Support Officer, British, currently living in Salford, Greater Manchester, UK.
Connor Walsh is a 28 year old Senior Customer Service Associate / Shift Lead (Retail Grocery Chain), American, currently living in Cleveland, Ohio, U

In [8]:
len(people)

12

In [9]:
# divide people in several groups of 5
people_groups = []
for i in range(0, len(people), 4):
    print(i)
    people_groups.append(people[i:i+4]
    )

len(people_groups)

0
4
8


3

In [10]:
people_groups

[[TinyPerson(name='Alan Merrick'),
  TinyPerson(name='Anthony Russo'),
  TinyPerson(name='Anya Calder-Mori'),
  TinyPerson(name='Barbara Jean Pratt')],
 [TinyPerson(name='Colin Arthur Matthews'),
  TinyPerson(name='Colin Murray'),
  TinyPerson(name='Connor Walsh'),
  TinyPerson(name='Darren McCall')],
 [TinyPerson(name='Dean Bartlett'),
  TinyPerson(name='Declan Blackwell'),
  TinyPerson(name='Edgar Milton Crane'),
  TinyPerson(name='Leonard Victor Hale')]]

In [11]:
# The experiment refers to customers

if experiment_runner.get_active_experiment() == "Control":
    for person in people:
        person.action_generator.enable_reasoning_step = False
        person.action_generator.enable_quality_checks = False

elif experiment_runner.get_active_experiment() == "Treatment":    
    for person in people:
       person.action_generator.enable_reasoning_step = False
       person.action_generator.enable_quality_checks = True
       person.action_generator.max_attempts = 2
       person.action_generator.enable_regeneration = True
       person.action_generator.quality_threshold = 5

## Proposals

In [12]:
proposals = [
    {"theme": "Daily Life and Convenience",
     "objective": "Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions."},

    {"theme": "Personal Growth and Wellbeing",
     "objective": "Generate concepts for products or experiences that support personal development, health, mental wellness, emotional care, or community connection."},

    {"theme": "Discovery and Exploration",
     "objective": "Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self."},

    {"theme": "Productivity and Resourcefulness",
     "objective": "Invent new tools, processes, or organizational systems that empower people or groups to achieve more, optimize resources, or collaborate effectively."},

    {"theme": "Creativity and Expression",
     "objective": "Design ideas for new products, platforms, or services that inspire creativity, foster expression, enhance artistic skills, or enable new forms of storytelling and communication."}
]

if not full_mode:
    proposals = proposals[:qty_proposals]

In [13]:
# divide the proposals in exactly two groups (half/half)
proposals_groups = []
proposals_groups.append(proposals[:len(proposals)//2])
proposals_groups.append(proposals[len(proposals)//2:])

proposals_groups

[[{'theme': 'Daily Life and Convenience',
   'objective': 'Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.'},
  {'theme': 'Personal Growth and Wellbeing',
   'objective': 'Generate concepts for products or experiences that support personal development, health, mental wellness, emotional care, or community connection.'}],
 [{'theme': 'Discovery and Exploration',
   'objective': 'Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.'},
  {'theme': 'Productivity and Resourcefulness',
   'objective': 'Invent new tools, processes, or organizational systems that empower people or groups to achieve more, optimize resources, or collaborate effectively.'},
  {'theme': 'Creativity and Expression',
   'objective': 'Design ideas for new products, platforms, or services that inspire creativity, foster expression, enhance art

## Auxiliary functions

In [14]:
def brainstorming_battery(agents, proposals, interventions, agent_propositions, environment_propositions, 
                          repetitions = 5, simulation_steps=10): 
    
    agent_propositions_scores = {}
    environment_propositions_scores = {}

    experiments_count = 0
    total_expected_experiments = len(proposals) * repetitions #* len(agents)

    # loop over proposals and repetitions
    for proposal in proposals:

        objective = proposal["objective"]
        theme = proposal["theme"]

        for i in range(repetitions):
            print("\n############## STARTING A NEW RESEARCH SESSION #################")
            print(f"Overall experiment number: {experiments_count+1} / {total_expected_experiments}")
            print(f"Discussion objective: {objective}")
            print(f"Trial number: {i+1}")
            print(f"Agents: {agents}")

            # clear the episodic memory of all agents
            for person in agents:
                person.clear_episodic_memory()

            world = TinyWorld(agents=agents, interventions=interventions)
            
            # Participants introduce themselves
            world.broadcast(f"""
                Hello everyone! Let's start by introducing ourselves, and mentioning problems we face in our daily personal
                and professional lives related to the following theme: {theme}
                
                Please:
                  - present yourself and your background;
                  - present some key personal problems related to the theme;
                  - present some key problems related to the theme that you face in your work;
                  - present some key problems related to the theme that you see in your industry as a whole.
                  
                Don't discuss solutions yet, just the problems you face and see others facing.
                """)
            world.run(1)
            
            # now to the brainstorming session itself
            world.broadcast(f"""
                Folks, your mission is to brainstorm {objective}. 
                Please follow these guidelines:
                  - give a unique and informative name to each idea you propose, so that it is easy to refer to it. Say it like "Idea name: '<name of the idea>'".;
                  - explain why you think it is a good idea, and what problem it solves, and how you feel about it;
                  - your ideas should be new complete, self-contained, products or services, not features for other existing products or services;
                  - think of creative ideas that would somehow help you in both in your personal and professional lives.
                  - create as many different and unique ideas as you can during the brainstorming session. Each idea must be **completely** different from the others 
                    (either by yourself or by others), and not just a variation of an existing idea.
                  - you should criticize each other's ideas, in order to make sure they are as
                    good as possible, but no more than once per idea.
                  - you should also provide suggestions for improvement to each other's ideas, in order to make them as good as possible, 
                    but no more than once per idea.
                  - regardless of critique or complement, you **must** primarily propose new ideas quickly, 
                    not just build on existing ones. 
                  - propose one idea at a time, instead of proposing multiple ideas at once, to allow appropriate discussion.
                  - you should **not** propose ideas that are too similar to each other, or to the ones already proposed by others.
                  - before saying anything, THINK deeply about yourself, your beliefs, interests, needs, life, etc., to come up with ideas that are
                    truly unique and different from the ones already proposed by others.
                   
                Please start the discussion now.
                """)
            world.run(simulation_steps)

            # extract and count ideas
            rapporteur = agents[0]  # the first agent is the rapporteur
            rapporteur.listen_and_act("Can you please consolidate the ideas that the group came up with? Provide a lot of details on each idea, and complement anything missing.")
            ideas = ResultsExtractor().extract_results_from_agent(rapporteur, 
                                    extraction_objective="Consolidates the ideas that the group came up with, explaining each idea as an item of a list." \
                                                        "Add information about: what problem the idea solves; to which target audience it is meant." \
                                                        "how is it different from competing, existing, products.", 
                                    situation="A focus group to brainstorm new product ideas.",
                                    fields= ["name", "description", "problem", "target_audience", "competition_analysis"],
                                    fields_hints={"ideas": "must be the root of the resulting dictionary."},)
            pprint(ideas)
            if "ideas_qty" not in environment_propositions_scores:
                environment_propositions_scores["ideas_qty"] = []
            if ideas is not None and "ideas" in ideas and isinstance(ideas["ideas"], list):
                environment_propositions_scores["ideas_qty"].append(len(ideas["ideas"]))

            # Evaluate environment propositions in parallel
            env_results = parallel_map_dict(
                environment_propositions,
                lambda item: item[1].copy().score(
                    world, 
                    claim_variables={"task_description": f"A brainstorming or focus group session was run about: {objective}."}, 
                    return_full_response=True
                )
            )
            
            # Process environment results
            for k, result in env_results.items():
                if k not in environment_propositions_scores:
                    environment_propositions_scores[k] = []
                environment_propositions_scores[k].append(result["value"])
                print("value: ", result["value"])
                print("justification: ", result["justification"])
                print("reasoning: ", result["reasoning"])

            # Evaluate agent propositions across all agents in parallel
            agent_results = parallel_map_cross(
                [agents, agent_propositions.items()],
                lambda agent, prop_item: (
                    prop_item[0],  # proposition key
                    prop_item[1].copy().score(agent, return_full_response=True)  # result
                )
            )
            
            # Process agent results
            for k, result in agent_results:
                if k not in agent_propositions_scores:
                    agent_propositions_scores[k] = []
                if result is not None:
                    agent_propositions_scores[k].append(result["value"])
                    print("value: ", result["value"])
                    print("justification: ", result["justification"])
                    print("reasoning: ", result["reasoning"])
                    print("\n\n")
                else:
                    print(f"*****WARNING:***** Agent did not respond to proposition {k}.")
            #
            ##for k, proposition in agent_propositions.items():
            ##    for person in world.agents:
            ##        result = proposition.copy().score(person, return_full_response=True)
            ##        
            ##        if k not in agent_propositions_scores:
            ##            agent_propositions_scores[k] = []
            ##        agent_propositions_scores[k].append(result["value"])
            ##
            ##        print("value: ", result["value"])
            ##        print("justification: ", result["justification"])
            ##        print("reasoning: ", result["reasoning"])
            ##        print("\n\n")
            ##
            
            experiments_count += 1
            print("\n\n")

    return agent_propositions_scores, environment_propositions_scores



## Perform experiment

In [15]:
agent_propositions_scores={}
environment_propositions_scores={}

In [16]:
def brainstorm(people, proposals=proposals):
    global agent_propositions_scores, environment_propositions_scores
    if not experiment_runner.has_finished_all_experiments():

        interventions = []
        #if experiment_runner.get_active_experiment() == "Treatment":
        #    interventions = \
        #        Intervention.create_for_each(people)\
        #            .set_functional_precondition(lambda target: target.actions_count >=7)\
        #            .set_textual_precondition(
        #                """
        #                AGENT IS NOT PROPOSING COMPLETELY NEW PRODUCT/SERVICE IDEAS ANYMORE:
        #                The last **entirely** new product/service idea proposed by this agent, if any, was proposed by him/her **more** than 5 of simulation events ago.
        #                That is to say, the agent has not proposed any new product/service idea in the last 5 of his/her simulation trajectory events.
        #                Additional features, variations of or other refinements to product/service ideas already proposed are NOT considered new!
#
        #                How to compute the steps gap:
        #                1. Determine the current next event number (N); and the last event number in which the agent proposed a new product/service idea (M).
        #                    This information can be found in the simulation trajectory.
        #                2. Compute the **difference** beteween the current next event number and the last event number in which the agent proposed a new product/service idea: D = N - M
        #                3. The proposition is true if, and only if, the difference D is **greater than** 5.
        #                """)\
        #            .set_effect(lambda target: target.think("""
        #                                                    I need to propose additional, **completelly** new and different, product/service ideas. This was part of the requirement for this session.
        #                                                    I will propose an entirely **new** idea now, I **cannot** repeat or refine previous ideas! I cannot make variations
        #                                                    of previous ideas (e.g., "XYZ for A", "XYZ for B", "XYZ for Z" are repetitive, there should be only one "XYZ"), 
        #                                                    I need to think of something **entirely** new and different.
        #                                                    To help me avoid repeating previous ideas, I'll now explicitly THINK about all the ideas already given by myself or
        #                                                    others, and then, based on that, I'll think again about a new unique idea.
        #                                                    """))

                                                            
        tmp_agent_propositions_scores, tmp_environment_propositions_scores = \
            brainstorming_battery(
                agents=people,
                proposals=proposals,
                interventions=interventions,    
                agent_propositions={
                    "Hard Persona Adherence": hard_persona_adherence,
                    "Self-consistency": self_consistency,
                    "Fluency": fluency
                },
                environment_propositions={
                    "Task Completion": task_completion,
                    "Divergence": divergence
                },
                repetitions=repetitions_per_task,
                simulation_steps=simulation_steps
            )

        pprint("NEW AGENT PROPOSITIONS SCORES")
        pprint(tmp_agent_propositions_scores)
        print("\n\n")
        pprint("NEW ENVIRONMENT PROPOSITIONS SCORES")
        pprint(tmp_environment_propositions_scores)

        # merge the scores lists
        agent_propositions_scores = merge_dicts_of_lists(tmp_agent_propositions_scores, agent_propositions_scores)
        environment_propositions_scores = merge_dicts_of_lists(tmp_environment_propositions_scores, environment_propositions_scores)

        return agent_propositions_scores, environment_propositions_scores

In [17]:
brainstorm(people_groups[0], proposals_groups[0]) if len(people_groups) > 0  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Alan Merrick'), TinyPerson(name='Anthony Russo'), TinyPerson(name='Anya Calder-Mori'), TinyPerson(name='Barbara Jean Pratt')]
2026-05-04 22:08:53,658 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 1] Running world simulation step 1 of 1.


───────────────────────────────────────────── TinyWorld 1 step 1 of 1 ─────────────────────────────────────────────

2026-05-04 22:08:53,721 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 1] No timedelta provided, so the datetime was not advanced.
2026-05-04 22:09:09,377 - ThreadPoolExecutor-0_1(32496) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-04 22:09:09,465 - ThreadPoolExecutor-0_2(54356) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-04 22:09:12,014 - ThreadPoolExecutor-0_1(32496) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-04 22:09:12,072 - ThreadPoolExecutor-0_2(54356) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-04 22:09:16,237 - ThreadPoolExecutor-0_3(35796) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-04 22:09:17,403 - ThreadPoolExecutor-0_3(35796) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-04 22:09:19,126 - ThreadPoolExecutor-0_0(48724) - t

───────────────────────────────────────────── TinyWorld 1 step 1 of 5 ─────────────────────────────────────────────

2026-05-04 22:13:27,477 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 1] No timedelta provided, so the datetime was not advanced.
2026-05-04 22:13:31,711 - ThreadPoolExecutor-1_2(35668) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-04 22:13:31,810 - ThreadPoolExecutor-1_2(35668) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-04 22:13:31,875 - ThreadPoolExecutor-1_3(39404) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-04 22:13:32,021 - ThreadPoolExecutor-1_3(39404) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-04 22:13:34,790 - ThreadPoolExecutor-1_0(56272) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-04 22:13:34,924 - ThreadPoolExecutor-1_0(56272) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-04 22:13:35,893 - ThreadPoolExecutor-1_1(30824) - t

...

───────────────────────────────────────────── TinyWorld 4 step 4 of 5 ─────────────────────────────────────────────

2026-05-05 00:02:30,804 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 4] No timedelta provided, so the datetime was not advanced.
2026-05-05 00:02:33,424 - ThreadPoolExecutor-28_0(50968) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:02:33,515 - ThreadPoolExecutor-28_0(50968) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 00:02:33,542 - ThreadPoolExecutor-28_3(42580) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:02:33,589 - ThreadPoolExecutor-28_2(53812) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:02:33,621 - ThreadPoolExecutor-28_1(53568) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:02:33,658 - ThreadPoolExecutor-28_3(42580) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 00:02:33,709 - ThreadPoolExecutor-28_1(53568) - tinytroupe - INFO - 

───────────────────────────────────────────── TinyWorld 4 step 5 of 5 ─────────────────────────────────────────────

2026-05-05 00:04:58,111 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 4] No timedelta provided, so the datetime was not advanced.
2026-05-05 00:05:00,456 - ThreadPoolExecutor-29_0(57936) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:05:00,488 - ThreadPoolExecutor-29_3(32160) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:05:00,551 - ThreadPoolExecutor-29_0(57936) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 00:05:00,582 - ThreadPoolExecutor-29_3(32160) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 00:05:00,669 - ThreadPoolExecutor-29_1(34676) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:05:00,676 - ThreadPoolExecutor-29_2(44592) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:05:00,761 - ThreadPoolExecutor-29_1(34676) - tinytroupe - INFO - 

({'Hard Persona Adherence': [7, 4, 0, 3, 3, 0, 1, 4, 0, 2, 4, 0, 1, 1, 3, 1],
  'Self-consistency': [9, 9, 7, 9, 7, 9, 7, 8, 9, 9, 9, 9, 9, 9, 9, 9],
  'Fluency': [7, 8, 9, 8, 8, 5, 9, 9, 7, 6, 8, 8, 7, 8, 8, 8]},
 {'ideas_qty': [2, 3, 3, 2],
  'Task Completion': [0, 2, 6, 6],
  'Divergence': [3, 1, 0, 0]})

In [18]:
brainstorm(people_groups[0], proposals_groups[1]) if len(people_groups) > 0  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Alan Merrick'), TinyPerson(name='Anthony Russo'), TinyPerson(name='Anya Calder-Mori'), TinyPerson(name='Barbara Jean Pratt')]
2026-05-05 00:17:05,427 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 5] Running world simulation step 1 of 1.


───────────────────────────────────────────── TinyWorld 5 step 1 of 1 ─────────────────────────────────────────────

2026-05-05 00:17:05,437 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 5] No timedelta provided, so the datetime was not advanced.
2026-05-05 00:17:07,763 - ThreadPoolExecutor-32_2(54736) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:17:07,784 - ThreadPoolExecutor-32_1(30896) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:17:07,803 - ThreadPoolExecutor-32_0(27880) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:17:07,822 - ThreadPoolExecutor-32_3(10124) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:17:07,846 - ThreadPoolExecutor-32_2(54736) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 00:17:07,869 - ThreadPoolExecutor-32_1(30896) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 00:17:07,875 - ThreadPoolExecutor-32_0(27880) - tinytroupe - INFO - 

───────────────────────────────────────────── TinyWorld 5 step 1 of 5 ─────────────────────────────────────────────

2026-05-05 00:19:37,405 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 5] No timedelta provided, so the datetime was not advanced.
2026-05-05 00:19:39,609 - ThreadPoolExecutor-33_1(55928) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:19:39,628 - ThreadPoolExecutor-33_2(8932) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:19:39,679 - ThreadPoolExecutor-33_3(55136) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:19:39,736 - ThreadPoolExecutor-33_0(25556) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 00:19:39,741 - ThreadPoolExecutor-33_1(55928) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 00:19:39,753 - ThreadPoolExecutor-33_2(8932) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 00:19:39,865 - ThreadPoolExecutor-33_3(55136) - tinytroupe - INFO - Wa

...

──────────────────────────────────────────── TinyWorld 10 step 4 of 5 ─────────────────────────────────────────────

2026-05-05 03:07:54,083 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 10] No timedelta provided, so the datetime was not advanced.
2026-05-05 03:07:56,299 - ThreadPoolExecutor-76_1(25016) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:07:56,317 - ThreadPoolExecutor-76_2(53464) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:07:56,372 - ThreadPoolExecutor-76_0(25768) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:07:56,399 - ThreadPoolExecutor-76_2(53464) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 03:07:56,401 - ThreadPoolExecutor-76_1(25016) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 03:07:56,446 - ThreadPoolExecutor-76_0(25768) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 03:07:56,446 - ThreadPoolExecutor-76_3(39

──────────────────────────────────────────── TinyWorld 10 step 5 of 5 ─────────────────────────────────────────────

2026-05-05 03:14:40,089 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 10] No timedelta provided, so the datetime was not advanced.
2026-05-05 03:14:42,431 - ThreadPoolExecutor-77_1(57316) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:14:42,467 - ThreadPoolExecutor-77_2(56948) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:14:42,518 - ThreadPoolExecutor-77_1(57316) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 03:14:42,556 - ThreadPoolExecutor-77_2(56948) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 03:14:42,750 - ThreadPoolExecutor-77_3(47360) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:14:42,818 - ThreadPoolExecutor-77_0(44964) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:14:42,881 - ThreadPoolExecutor-77_3(47360) - tinytroupe - INFO -

({'Hard Persona Adherence': [7,
   4,
   0,
   3,
   3,
   0,
   1,
   4,
   0,
   2,
   4,
   0,
   1,
   1,
   3,
   1,
   3,
   1,
   0,
   2,
   0,
   1,
   0,
   3,
   4,
   0,
   1,
   5,
   3,
   0,
   1,
   2,
   3,
   7,
   0,
   3,
   0,
   1,
   4,
   3],
  'Self-consistency': [9,
   9,
   7,
   9,
   7,
   9,
   7,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   6,
   9,
   9],
  'Fluency': [7,
   8,
   9,
   8,
   8,
   5,
   9,
   9,
   7,
   6,
   8,
   8,
   7,
   8,
   8,
   8,
   8,
   8,
   9,
   8,
   7,
   7,
   9,
   8,
   8,
   8,
   9,
   8,
   7,
   8,
   7,
   8,
   9,
   7,
   9,
   9,
   8,
   7,
   9,
   9]},
 {'ideas_qty': [2, 3, 3, 2, 5, 4, 2, 3, 3, 2],
  'Task Completion': [0, 2, 6, 6, 0, 6, 7, 7, 8, 4],
  'Divergence': [3, 1, 0, 0, 1, 0, 8, 2, 8, 0]})

In [19]:
brainstorm(people_groups[1], proposals_groups[0]) if len(people_groups) > 1  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Colin Arthur Matthews'), TinyPerson(name='Colin Murray'), TinyPerson(name='Connor Walsh'), TinyPerson(name='Darren McCall')]
2026-05-05 03:33:25,540 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 11] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 11 step 1 of 1 ─────────────────────────────────────────────

2026-05-05 03:33:25,548 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 11] No timedelta provided, so the datetime was not advanced.
2026-05-05 03:33:27,695 - ThreadPoolExecutor-80_0(11936) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:33:27,750 - ThreadPoolExecutor-80_0(11936) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 03:33:27,783 - ThreadPoolExecutor-80_3(38988) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:33:27,836 - ThreadPoolExecutor-80_1(39872) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:33:27,847 - ThreadPoolExecutor-80_2(33540) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:33:27,882 - ThreadPoolExecutor-80_3(38988) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 03:33:27,916 - ThreadPoolExecutor-80_1(39872) - tinytroupe - INFO -

──────────────────────────────────────────── TinyWorld 11 step 1 of 5 ─────────────────────────────────────────────

2026-05-05 03:37:58,032 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 11] No timedelta provided, so the datetime was not advanced.
2026-05-05 03:38:00,390 - ThreadPoolExecutor-81_0(43408) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:38:00,471 - ThreadPoolExecutor-81_3(35320) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:38:00,482 - ThreadPoolExecutor-81_0(43408) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 03:38:00,554 - ThreadPoolExecutor-81_3(35320) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 03:38:00,575 - ThreadPoolExecutor-81_2(29244) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:38:00,596 - ThreadPoolExecutor-81_1(31612) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 03:38:00,655 - ThreadPoolExecutor-81_2(29244) - tinytroupe - INFO -

...

──────────────────────────────────────────── TinyWorld 14 step 4 of 5 ─────────────────────────────────────────────

2026-05-05 05:40:29,079 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 14] No timedelta provided, so the datetime was not advanced.
2026-05-05 05:40:31,966 - ThreadPoolExecutor-108_0(57568) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:40:32,040 - ThreadPoolExecutor-108_0(57568) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 05:40:32,041 - ThreadPoolExecutor-108_1(20688) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:40:32,122 - ThreadPoolExecutor-108_1(20688) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 05:40:32,207 - ThreadPoolExecutor-108_3(39420) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:40:32,214 - ThreadPoolExecutor-108_2(39184) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:40:32,280 - ThreadPoolExecutor-108_2(39184) - tinytroupe -

──────────────────────────────────────────── TinyWorld 14 step 5 of 5 ─────────────────────────────────────────────

2026-05-05 05:43:10,937 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 14] No timedelta provided, so the datetime was not advanced.
2026-05-05 05:43:13,175 - ThreadPoolExecutor-109_0(28436) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:43:13,204 - ThreadPoolExecutor-109_1(11976) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:43:13,245 - ThreadPoolExecutor-109_0(28436) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 05:43:13,262 - ThreadPoolExecutor-109_2(53880) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:43:13,287 - ThreadPoolExecutor-109_1(11976) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 05:43:13,293 - ThreadPoolExecutor-109_3(56564) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:43:13,341 - ThreadPoolExecutor-109_2(53880) - tinytroupe -

({'Hard Persona Adherence': [7,
   4,
   0,
   3,
   3,
   0,
   1,
   4,
   0,
   2,
   4,
   0,
   1,
   1,
   3,
   1,
   3,
   1,
   0,
   2,
   0,
   1,
   0,
   3,
   4,
   0,
   1,
   5,
   3,
   0,
   1,
   2,
   3,
   7,
   0,
   3,
   0,
   1,
   4,
   3,
   3,
   3,
   4,
   3,
   2,
   2,
   2,
   0,
   4,
   2,
   2,
   2,
   4,
   3,
   0,
   7],
  'Self-consistency': [9,
   9,
   7,
   9,
   7,
   9,
   7,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   6,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   8,
   9,
   9,
   9,
   5,
   9,
   9,
   9,
   9,
   7],
  'Fluency': [7,
   8,
   9,
   8,
   8,
   5,
   9,
   9,
   7,
   6,
   8,
   8,
   7,
   8,
   8,
   8,
   8,
   8,
   9,
   8,
   7,
   7,
   9,
   8,
   8,
   8,
   9,
   8,
   7,
   8,
   7,
   8,
   9,
   7,
   9,
   9,
   8,
   7,
   9,
   9,
   9,
   8,
   9,
   9,
   9,

In [20]:
brainstorm(people_groups[1], proposals_groups[1]) if len(people_groups) > 1  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Colin Arthur Matthews'), TinyPerson(name='Colin Murray'), TinyPerson(name='Connor Walsh'), TinyPerson(name='Darren McCall')]
2026-05-05 05:54:07,408 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 15] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 15 step 1 of 1 ─────────────────────────────────────────────

2026-05-05 05:54:07,417 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 15] No timedelta provided, so the datetime was not advanced.
2026-05-05 05:54:10,152 - ThreadPoolExecutor-112_2(58984) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:54:10,221 - ThreadPoolExecutor-112_2(58984) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 05:54:10,270 - ThreadPoolExecutor-112_3(31328) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:54:10,363 - ThreadPoolExecutor-112_1(52548) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:54:10,398 - ThreadPoolExecutor-112_3(31328) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 05:54:10,454 - ThreadPoolExecutor-112_0(42004) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:54:10,486 - ThreadPoolExecutor-112_1(52548) - tinytroupe -

──────────────────────────────────────────── TinyWorld 15 step 1 of 5 ─────────────────────────────────────────────

2026-05-05 05:56:45,285 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 15] No timedelta provided, so the datetime was not advanced.
2026-05-05 05:56:47,033 - ThreadPoolExecutor-113_0(52568) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:56:47,049 - ThreadPoolExecutor-113_3(43500) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:56:47,081 - ThreadPoolExecutor-113_2(43992) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:56:47,085 - ThreadPoolExecutor-113_1(58872) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 05:56:47,110 - ThreadPoolExecutor-113_0(52568) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 05:56:47,116 - ThreadPoolExecutor-113_3(43500) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 05:56:47,143 - ThreadPoolExecutor-113_2(43992) - tinytroupe -

...

──────────────────────────────────────────── TinyWorld 20 step 4 of 5 ─────────────────────────────────────────────

2026-05-05 08:55:32,619 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 20] No timedelta provided, so the datetime was not advanced.
2026-05-05 08:55:35,772 - ThreadPoolExecutor-156_0(15432) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 08:55:35,846 - ThreadPoolExecutor-156_3(11060) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 08:55:35,877 - ThreadPoolExecutor-156_0(15432) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 08:55:35,878 - ThreadPoolExecutor-156_1(14064) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 08:55:35,946 - ThreadPoolExecutor-156_3(11060) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 08:55:35,947 - ThreadPoolExecutor-156_2(26692) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 08:55:35,977 - ThreadPoolExecutor-156_1(14064) - tinytroupe -

──────────────────────────────────────────── TinyWorld 20 step 5 of 5 ─────────────────────────────────────────────

2026-05-05 08:59:20,208 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 20] No timedelta provided, so the datetime was not advanced.
2026-05-05 08:59:23,339 - ThreadPoolExecutor-157_0(39496) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 08:59:23,422 - ThreadPoolExecutor-157_0(39496) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 08:59:23,441 - ThreadPoolExecutor-157_3(58056) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 08:59:23,479 - ThreadPoolExecutor-157_2(47792) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 08:59:23,530 - ThreadPoolExecutor-157_3(58056) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 08:59:23,580 - ThreadPoolExecutor-157_1(30632) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 08:59:23,605 - ThreadPoolExecutor-157_2(47792) - tinytroupe -

({'Hard Persona Adherence': [7,
   4,
   0,
   3,
   3,
   0,
   1,
   4,
   0,
   2,
   4,
   0,
   1,
   1,
   3,
   1,
   3,
   1,
   0,
   2,
   0,
   1,
   0,
   3,
   4,
   0,
   1,
   5,
   3,
   0,
   1,
   2,
   3,
   7,
   0,
   3,
   0,
   1,
   4,
   3,
   3,
   3,
   4,
   3,
   2,
   2,
   2,
   0,
   4,
   2,
   2,
   2,
   4,
   3,
   0,
   7,
   0,
   3,
   3,
   4,
   2,
   0,
   3,
   2,
   3,
   5,
   3,
   2,
   4,
   1,
   1,
   3,
   1,
   0,
   3,
   3,
   0,
   0,
   3,
   3],
  'Self-consistency': [9,
   9,
   7,
   9,
   7,
   9,
   7,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   6,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   8,
   9,
   9,
   9,
   5,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   6,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   7,
   6,
   9,
   8,
   7,
   7,
   6,
   9,

In [21]:
brainstorm(people_groups[2], proposals_groups[0]) if len(people_groups) > 2  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Dean Bartlett'), TinyPerson(name='Declan Blackwell'), TinyPerson(name='Edgar Milton Crane'), TinyPerson(name='Leonard Victor Hale')]
2026-05-05 09:12:12,933 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 21] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 21 step 1 of 1 ─────────────────────────────────────────────

2026-05-05 09:12:12,942 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 21] No timedelta provided, so the datetime was not advanced.
2026-05-05 09:12:15,206 - ThreadPoolExecutor-160_2(33304) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 09:12:15,266 - ThreadPoolExecutor-160_1(50596) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 09:12:15,283 - ThreadPoolExecutor-160_3(27072) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 09:12:15,310 - ThreadPoolExecutor-160_0(37292) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 09:12:15,323 - ThreadPoolExecutor-160_2(33304) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 09:12:15,364 - ThreadPoolExecutor-160_1(50596) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 09:12:15,380 - ThreadPoolExecutor-160_3(27072) - tinytroupe -

──────────────────────────────────────────── TinyWorld 21 step 1 of 5 ─────────────────────────────────────────────

2026-05-05 09:15:03,793 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 21] No timedelta provided, so the datetime was not advanced.
2026-05-05 09:15:05,988 - ThreadPoolExecutor-161_2(31284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 09:15:06,053 - ThreadPoolExecutor-161_1(15796) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 09:15:06,115 - ThreadPoolExecutor-161_2(31284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 09:15:06,152 - ThreadPoolExecutor-161_1(15796) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 09:15:06,185 - ThreadPoolExecutor-161_0(54804) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 09:15:06,217 - ThreadPoolExecutor-161_3(15468) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 09:15:06,298 - ThreadPoolExecutor-161_0(54804) - tinytroupe -

...

──────────────────────────────────────────── TinyWorld 24 step 4 of 5 ─────────────────────────────────────────────

2026-05-05 11:27:58,400 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 24] No timedelta provided, so the datetime was not advanced.
2026-05-05 11:28:02,352 - ThreadPoolExecutor-188_0(54648) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 11:28:02,425 - ThreadPoolExecutor-188_1(19452) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 11:28:02,528 - ThreadPoolExecutor-188_0(54648) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 11:28:02,573 - ThreadPoolExecutor-188_1(19452) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 11:28:04,261 - ThreadPoolExecutor-188_3(55812) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 11:28:04,557 - ThreadPoolExecutor-188_3(55812) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 11:28:04,832 - ThreadPoolExecutor-1

──────────────────────────────────────────── TinyWorld 24 step 5 of 5 ─────────────────────────────────────────────

2026-05-05 11:34:11,468 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 24] No timedelta provided, so the datetime was not advanced.
2026-05-05 11:34:15,053 - ThreadPoolExecutor-189_0(57068) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 11:34:15,138 - ThreadPoolExecutor-189_3(47292) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 11:34:15,154 - ThreadPoolExecutor-189_0(57068) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 11:34:15,237 - ThreadPoolExecutor-189_1(59076) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 11:34:15,272 - ThreadPoolExecutor-189_3(47292) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 11:34:15,339 - ThreadPoolExecutor-189_1(59076) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 11:34:15,345 - ThreadPoolExecutor-1

({'Hard Persona Adherence': [7,
   4,
   0,
   3,
   3,
   0,
   1,
   4,
   0,
   2,
   4,
   0,
   1,
   1,
   3,
   1,
   3,
   1,
   0,
   2,
   0,
   1,
   0,
   3,
   4,
   0,
   1,
   5,
   3,
   0,
   1,
   2,
   3,
   7,
   0,
   3,
   0,
   1,
   4,
   3,
   3,
   3,
   4,
   3,
   2,
   2,
   2,
   0,
   4,
   2,
   2,
   2,
   4,
   3,
   0,
   7,
   0,
   3,
   3,
   4,
   2,
   0,
   3,
   2,
   3,
   5,
   3,
   2,
   4,
   1,
   1,
   3,
   1,
   0,
   3,
   3,
   0,
   0,
   3,
   3,
   2,
   4,
   3,
   3,
   0,
   5,
   1,
   2,
   1,
   5,
   1,
   4,
   3,
   3,
   1,
   3],
  'Self-consistency': [9,
   9,
   7,
   9,
   7,
   9,
   7,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   6,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   8,
   9,
   9,
   9,
   5,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   6,
   9,
   9,
   9,
   9,

In [22]:
brainstorm(people_groups[2], proposals_groups[1]) if len(people_groups) > 2  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Dean Bartlett'), TinyPerson(name='Declan Blackwell'), TinyPerson(name='Edgar Milton Crane'), TinyPerson(name='Leonard Victor Hale')]
2026-05-05 11:50:28,016 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 25] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 25 step 1 of 1 ─────────────────────────────────────────────

2026-05-05 11:50:28,030 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 25] No timedelta provided, so the datetime was not advanced.
2026-05-05 11:50:31,995 - ThreadPoolExecutor-192_1(41544) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 11:50:32,063 - ThreadPoolExecutor-192_2(12516) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 11:50:32,102 - ThreadPoolExecutor-192_1(41544) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 11:50:32,156 - ThreadPoolExecutor-192_2(12516) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 11:50:33,875 - ThreadPoolExecutor-192_3(58520) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 11:50:34,001 - ThreadPoolExecutor-192_3(58520) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 11:50:34,003 - ThreadPoolExecutor-1

──────────────────────────────────────────── TinyWorld 25 step 1 of 5 ─────────────────────────────────────────────

2026-05-05 11:56:22,771 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 25] No timedelta provided, so the datetime was not advanced.
2026-05-05 11:56:28,143 - ThreadPoolExecutor-193_1(47544) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 11:56:28,281 - ThreadPoolExecutor-193_2(2916) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 11:56:28,323 - ThreadPoolExecutor-193_1(47544) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 11:56:28,412 - ThreadPoolExecutor-193_2(2916) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 11:56:30,459 - ThreadPoolExecutor-193_0(39944) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 11:56:30,505 - ThreadPoolExecutor-193_3(56168) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 11:56:30,673 - ThreadPoolExecutor-193_3(56168) - tinytroupe - I

...

──────────────────────────────────────────── TinyWorld 30 step 4 of 5 ─────────────────────────────────────────────

2026-05-05 16:06:59,056 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 30] No timedelta provided, so the datetime was not advanced.
2026-05-05 16:07:02,250 - ThreadPoolExecutor-236_2(50292) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 16:07:02,264 - ThreadPoolExecutor-236_1(45324) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 16:07:02,344 - ThreadPoolExecutor-236_0(56112) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 16:07:02,355 - ThreadPoolExecutor-236_1(45324) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 16:07:02,371 - ThreadPoolExecutor-236_2(50292) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 16:07:02,390 - ThreadPoolExecutor-236_3(10792) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 16:07:02,428 - ThreadPoolExecutor-236_0(56112) - tinytroupe -

──────────────────────────────────────────── TinyWorld 30 step 5 of 5 ─────────────────────────────────────────────

2026-05-05 16:09:24,470 - MainThread(44240) - tinytroupe - INFO - [TinyWorld 30] No timedelta provided, so the datetime was not advanced.
2026-05-05 16:09:26,917 - ThreadPoolExecutor-237_2(51216) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 16:09:26,964 - ThreadPoolExecutor-237_3(15796) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 16:09:26,986 - ThreadPoolExecutor-237_1(20644) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 16:09:27,006 - ThreadPoolExecutor-237_0(43380) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-05 16:09:27,030 - ThreadPoolExecutor-237_2(51216) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 16:09:27,040 - ThreadPoolExecutor-237_3(15796) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-05 16:09:27,068 - ThreadPoolExecutor-237_1(20644) - tinytroupe -

({'Hard Persona Adherence': [7,
   4,
   0,
   3,
   3,
   0,
   1,
   4,
   0,
   2,
   4,
   0,
   1,
   1,
   3,
   1,
   3,
   1,
   0,
   2,
   0,
   1,
   0,
   3,
   4,
   0,
   1,
   5,
   3,
   0,
   1,
   2,
   3,
   7,
   0,
   3,
   0,
   1,
   4,
   3,
   3,
   3,
   4,
   3,
   2,
   2,
   2,
   0,
   4,
   2,
   2,
   2,
   4,
   3,
   0,
   7,
   0,
   3,
   3,
   4,
   2,
   0,
   3,
   2,
   3,
   5,
   3,
   2,
   4,
   1,
   1,
   3,
   1,
   0,
   3,
   3,
   0,
   0,
   3,
   3,
   2,
   4,
   3,
   3,
   0,
   5,
   1,
   2,
   1,
   5,
   1,
   4,
   3,
   3,
   1,
   3,
   0,
   4,
   0,
   2,
   2,
   4,
   3,
   3,
   0,
   1,
   4,
   4,
   2,
   3,
   3,
   3,
   3,
   5,
   3,
   4,
   5,
   3,
   1,
   0],
  'Self-consistency': [9,
   9,
   7,
   9,
   7,
   9,
   7,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   6,
   9,

In [23]:
brainstorm(people_groups[3], proposals_groups[0]) if len(people_groups) > 3  and len(proposals_groups) > 0 else None

In [24]:
brainstorm(people_groups[3], proposals_groups[1]) if len(people_groups) > 3  and len(proposals_groups) > 1 else None

In [25]:
brainstorm(people_groups[4], proposals_groups[0]) if len(people_groups) > 4  and len(proposals_groups) > 0 else None

In [26]:
brainstorm(people_groups[4], proposals_groups[1]) if len(people_groups) > 4  and len(proposals_groups) > 1 else None

## Extract results and analyze

In [27]:
if experiment_runner.get_active_experiment() in ["Control", "Treatment"]:
    combined_scores = {**agent_propositions_scores, **environment_propositions_scores}
    experiment_runner.add_experiment_results(combined_scores, experiment_name=experiment_runner.get_active_experiment()) 
    
    plot_scores(combined_scores)

else:
    print("Experiment finished. No more experiments to run.")

{'Divergence': [3,
                1,
                0,
                0,
                1,
                0,
                8,
                2,
                8,
                0,
                4,
                1,
                0,
                1,
                2,
                1,
                0,
                1,
                2,
                3,
                0,
                0,
                0,
                0,
                0,
                1,
                7,
                0,
                2,
                3],
 'Fluency': [7,
             8,
             9,
             8,
             8,
             5,
             9,
             9,
             7,
             6,
             8,
             8,
             7,
             8,
             8,
             8,
             8,
             8,
             9,
             8,
             7,
             7,
             9,
             8,
             8,
             8,
             

,Proposition,Average Score,Standard Deviation,Count
0,Hard Persona Adherence,2.325000,1.666085,120.0
1,Self-consistency,8.216667,1.730514,120.0
2,Fluency,62.133333,593.462195,120.0
3,ideas_qty,2.833333,0.916831,24.0
4,Task Completion,5.433333,3.359837,30.0
5,Divergence,1.700000,2.321563,30.0


In [28]:
if experiment_runner.has_finished_all_experiments():
    print("All experiments have been finished.")
    print(f"STATISTICTS: Control vs")
    pprint(experiment_runner.run_statistical_tests(control_experiment_name='Control'))

    # plot scores of both experiments
    experiment_control_scores = experiment_runner.get_experiment_results("Control")
    experiment_treatment_scores = experiment_runner.get_experiment_results("Treatment")
    
    
    plot_scores(experiment_control_scores)
    plot_scores(experiment_treatment_scores)

else:
    print("Not all experiments have been finished. RESTART AND RERUN.")

Not all experiments have been finished. RESTART AND RERUN.


In [29]:
experiment_runner.finish_active_experiment()

2026-05-05 16:26:38,865 - MainThread(44240) - tinytroupe - INFO - Experiment 'Treatment' marked as finished.
2026-05-05 16:26:38,869 - MainThread(44240) - tinytroupe - INFO - All experiments have been finished.


True